# NMT MarianMT — Spanish–English Translation (**SOLVED**)

**Task:** Neural Machine Translation (NMT), translating *Spanish → English* (or the reverse) using a pre-trained Transformer encoder–decoder.

**Model:** `Helsinki-NLP/opus-mt-es-en` (MarianMT) — encoder–decoder Transformer.

---

Esta es la **versión resuelta** del notebook. Incluye todas las celdas de código completas y comentadas.
Puedes usarlo como referencia después de intentar la versión **STUDENT** por tu cuenta.

---

## Objetivos del notebook

En este notebook vas a:

1. Cargar un modelo de traducción automática basado en Transformer (MarianMT) usando `transformers`.
2. Traducir frases sencillas entre español e inglés.
3. Experimentar con distintas estrategias de decodificación:
   - Greedy (búsqueda codiciosa).
   - Beam search con distintos `num_beams`.
   - Sampling (top-k, top-p, temperatura).
4. Implementar una mini-métrica de evaluación basada en **solapamiento de n-gramas** (aprox. tipo BLEU muy simplificado).
5. Analizar algunos **errores típicos** de NMT (under-translation, over-translation, errores léxicos/estilísticos).

---

## Qué se asume que ya sabes

- Arquitectura Transformer encoder–decoder (Tema 3).
- Conceptos básicos de NMT (codificador, decodificador, atención).
- Nociones sobre estrategias de decodificación: greedy, beam search, sampling.
- Python y nociones de PyTorch.


## 1) (Opcional) Instalación / actualización de librerías

Si estás en Google Colab o en un entorno donde no tienes `transformers`/`sentencepiece` instalados (o quieres actualizar), puedes usar la celda de abajo.

- Si ya tienes el entorno preparado (por ejemplo, viene dado por el profesor), puedes **saltar esta celda**.


In [1]:
# (Opcional) Instalación / actualización de librerías en Colab.
# Normalmente bastará con ejecutar esta celda una vez al principio del notebook.

# import sys
# pip = f"{sys.executable} -m pip"
# !{pip} install -q "transformers>=4.40.0" sentencepiece

## 2) Imports y configuración básica

In [2]:
import torch
from transformers import MarianMTModel, MarianTokenizer
import numpy as np
from collections import Counter
import re

# Detectamos dispositivo disponible (GPU si existe)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Fijamos semillas para cierta reproducibilidad
torch.manual_seed(42)
if device == "cuda":
    torch.cuda.manual_seed_all(42)
np.random.seed(42)

Using device: cuda


## 3) Carga del modelo MarianMT y del tokenizador

En esta práctica usaremos por defecto el modelo **es → en** (`Helsinki-NLP/opus-mt-es-en`), pero puedes cambiar fácilmente el par de lenguas.


In [ ]:
# Definimos el par de idiomas
src_lang = "es"
tgt_lang = "en"

model_name = f"Helsinki-NLP/opus-mt-{src_lang}-{tgt_lang}"
print("Using model:", model_name)

# Carga de tokenizador y modelo
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

print("Model and tokenizer loaded.")

Using model: Helsinki-NLP/opus-mt-es-en


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model and tokenizer loaded.


Using model: Helsinki-NLP/opus-mt-es-en


KeyboardInterrupt: 

## 4) Explorando el tokenizador (subword units)

Vamos a ver cómo el tokenizador de MarianMT segmenta frases en subpalabras (BPE / SentencePiece).


In [ ]:
example_sentence = "Los modelos Transformer han revolucionado el procesamiento del lenguaje natural."
encoded = tokenizer(example_sentence, return_tensors="pt")
print("Input IDs:", encoded["input_ids"])

ids = encoded["input_ids"][0]
subwords = tokenizer.convert_ids_to_tokens(ids)
print("Subwords:")
for i, (idx, sw) in enumerate(zip(ids.tolist(), subwords)):
    print(f"{i:2d} -> {idx:6d} -> {sw}")

decoded = tokenizer.decode(ids, skip_special_tokens=True)
print("\nDecoded again:", decoded)

# Otro ejemplo con palabras técnicas o poco frecuentes
example_sentence_2 = "Las metaheurísticas multiobjetivo se usan en optimización combinatoria."
encoded2 = tokenizer(example_sentence_2, return_tensors="pt")
ids2 = encoded2["input_ids"][0]
subwords2 = tokenizer.convert_ids_to_tokens(ids2)
print("\nSegundo ejemplo:")
for i, (idx, sw) in enumerate(zip(ids2.tolist(), subwords2)):
    print(f"{i:2d} -> {idx:6d} -> {sw}")

Input IDs: tensor([[  131,  4730,  6179, 24062,    91,   159,   242, 23708,  1397,    14,
         11136,    19,  9100,   869,     3,     0]])
Subwords:
 0 ->    131 -> ▁Los
 1 ->   4730 -> ▁modelos
 2 ->   6179 -> ▁Trans
 3 ->  24062 -> forme
 4 ->     91 -> r
 5 ->    159 -> ▁han
 6 ->    242 -> ▁re
 7 ->  23708 -> volucion
 8 ->   1397 -> ado
 9 ->     14 -> ▁el
10 ->  11136 -> ▁procesamiento
11 ->     19 -> ▁del
12 ->   9100 -> ▁lenguaje
13 ->    869 -> ▁natural
14 ->      3 -> .
15 ->      0 -> </s>

Decoded again: Los modelos Transformer han revolucionado el procesamiento del lenguaje natural.

Segundo ejemplo:
 0 ->    219 -> ▁Las
 1 ->   6257 -> ▁meta
 2 ->   2981 -> he
 3 ->   1830 -> ur
 4 ->  18226 -> ísticas
 5 ->   2903 -> ▁multi
 6 ->  42697 -> objetivo
 7 ->     26 -> ▁se
 8 ->  12345 -> ▁usan
 9 ->     12 -> ▁en
10 ->  36766 -> ▁optimización
11 ->  13264 -> ▁combina
12 ->   6477 -> toria
13 ->      3 -> .
14 ->      0 -> </s>


## 5) Mini-dataset artesanal de frases (ES→EN)

Definimos un pequeño conjunto de frases de ejemplo con sus traducciones de referencia en inglés.


In [ ]:
pairs = [
    {
        "src": "Este es un curso de Procesamiento del Lenguaje Natural.",
        "ref": "This is a Natural Language Processing course."
    },
    {
        "src": "Los modelos Transformer han revolucionado el PLN.",
        "ref": "Transformer models have revolutionized NLP."
    },
    {
        "src": "La evaluación automática de la traducción sigue siendo un desafío.",
        "ref": "Automatic evaluation of translation is still a challenge."
    },
    {
        "src": "Las metaheurísticas multiobjetivo permiten explorar compromisos entre varios objetivos.",
        "ref": "Multi-objective metaheuristics allow exploring trade-offs between several objectives."
    },
    {
        "src": "Los sistemas de traducción automática neuronal requieren grandes cantidades de datos paralelos.",
        "ref": "Neural machine translation systems require large amounts of parallel data."
    },
]

for i, ex in enumerate(pairs):
    print(f"Ejemplo {i+1}")
    print("SRC:", ex["src"])
    print("REF:", ex["ref"])
    print()

Ejemplo 1
SRC: Este es un curso de Procesamiento del Lenguaje Natural.
REF: This is a Natural Language Processing course.

Ejemplo 2
SRC: Los modelos Transformer han revolucionado el PLN.
REF: Transformer models have revolutionized NLP.

Ejemplo 3
SRC: La evaluación automática de la traducción sigue siendo un desafío.
REF: Automatic evaluation of translation is still a challenge.

Ejemplo 4
SRC: Las metaheurísticas multiobjetivo permiten explorar compromisos entre varios objetivos.
REF: Multi-objective metaheuristics allow exploring trade-offs between several objectives.

Ejemplo 5
SRC: Los sistemas de traducción automática neuronal requieren grandes cantidades de datos paralelos.
REF: Neural machine translation systems require large amounts of parallel data.



## 6) Función de traducción con parámetros de decodificación

Implementamos una función `translate` que nos permitirá reutilizar fácilmente el modelo con distintos parámetros.


In [ ]:
def translate(
    sentences,
    num_beams: int = 1,
    max_length: int = 64,
    do_sample: bool = False,
    top_k: int | None = None,
    top_p: float | None = None,
    temperature: float = 1.0,
):
    """Traduce una o varias frases usando MarianMT.

    Args:
        sentences: string o lista de strings con las frases origen.
        num_beams: tamaño del beam (1 = greedy).
        max_length: longitud máxima de la secuencia generada.
        do_sample: si True, activa muestreo estocástico.
        top_k, top_p, temperature: parámetros de sampling.

    Returns:
        Lista de strings con las traducciones generadas.
    """
    if isinstance(sentences, str):
        sentences = [sentences]

    # Tokenización y paso a dispositivo
    encodings = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    # Generación
    with torch.no_grad():
        generated_ids = model.generate(
            **encodings,
            num_beams=num_beams,
            max_length=max_length,
            do_sample=do_sample,
            top_k=top_k,
            top_p=top_p,
            temperature=temperature
        )

    # Decodificación
    outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    return outputs

# Prueba rápida
print(translate("Los modelos de lenguaje grandes pueden generar texto coherente.")[0])

Large language models can generate coherent text.


## 7) Experimento 1 — Greedy decoding (num_beams = 1)

Probamos primero la decodificación codiciosa (greedy), que corresponde a `num_beams=1`.


In [ ]:
src_sentences = [ex["src"] for ex in pairs]
refs = [ex["ref"] for ex in pairs]

translations_greedy = translate(src_sentences, num_beams=1)

for src, hyp, ref in zip(src_sentences, translations_greedy, refs):
    print("SRC:", src)
    print("HYP (greedy):", hyp)
    print("REF:", ref)
    print()

SRC: Este es un curso de Procesamiento del Lenguaje Natural.
HYP (greedy): This is a Natural Language Processing course.
REF: This is a Natural Language Processing course.

SRC: Los modelos Transformer han revolucionado el PLN.
HYP (greedy): The Transformer models have revolutionized the PLN.
REF: Transformer models have revolutionized NLP.

SRC: La evaluación automática de la traducción sigue siendo un desafío.
HYP (greedy): Automatic evaluation of translation remains a challenge.
REF: Automatic evaluation of translation is still a challenge.

SRC: Las metaheurísticas multiobjetivo permiten explorar compromisos entre varios objetivos.
HYP (greedy): Multi-objective meta-heuristics allow us to explore compromises between various objectives.
REF: Multi-objective metaheuristics allow exploring trade-offs between several objectives.

SRC: Los sistemas de traducción automática neuronal requieren grandes cantidades de datos paralelos.
HYP (greedy): Neuronal machine translation systems requir

## 8) Experimento 2 — Beam search con distintos `num_beams`

Probamos ahora con `num_beams = 4` (y opcionalmente 8) para ver si mejoran las traducciones.


In [ ]:
translations_beam4 = translate(src_sentences, num_beams=4)
# Opcionalmente, podríamos probar también num_beams=8:
# translations_beam8 = translate(src_sentences, num_beams=8)

for src, hyp_g, hyp_b4, ref in zip(src_sentences, translations_greedy, translations_beam4, refs):
    print("SRC:", src)
    print("HYP (greedy):", hyp_g)
    print("HYP (beam=4):", hyp_b4)
    print("REF:", ref)
    print("-" * 80)

SRC: Este es un curso de Procesamiento del Lenguaje Natural.
HYP (greedy): This is a Natural Language Processing course.
HYP (beam=4): This is a Natural Language Processing course.
REF: This is a Natural Language Processing course.
--------------------------------------------------------------------------------
SRC: Los modelos Transformer han revolucionado el PLN.
HYP (greedy): The Transformer models have revolutionized the PLN.
HYP (beam=4): The Transformer models have revolutionized the PLN.
REF: Transformer models have revolutionized NLP.
--------------------------------------------------------------------------------
SRC: La evaluación automática de la traducción sigue siendo un desafío.
HYP (greedy): Automatic evaluation of translation remains a challenge.
HYP (beam=4): The automatic evaluation of translation remains a challenge.
REF: Automatic evaluation of translation is still a challenge.
--------------------------------------------------------------------------------
SRC: Las

## 9) Experimento 3 — Sampling (top-k, top-p, temperatura)

Exploramos ahora sampling para incrementar la diversidad de traducciones.


In [ ]:
sample_sentence = "Los modelos de lenguaje grandes pueden generar texto coherente."

for temp in [0.3, 0.7, 1.2]:
    print("=== temperature =", temp, "===")
    hyp = translate(
        sample_sentence,
        num_beams=1,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=temp
    )[0]
    print("HYP:", hyp)
    print()

=== temperature = 0.3 ===
HYP: Large language models can generate coherent text.

=== temperature = 0.7 ===
HYP: Large language models can generate coherent text.

=== temperature = 1.2 ===
HYP: Large language models can generate consistent text.



## 10) Mini-métrica de evaluación basada en n-gramas

Implementamos una métrica muy simple basada en solapamiento de n-gramas entre la traducción generada (HYP) y la referencia (REF).


In [ ]:
def tokenize_simple(text: str):
    """Tokenización muy simple: minúsculas + eliminación de puntuación básica."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9áéíóúüñ ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def ngram_overlap(hyp: str, ref: str, n: int = 1) -> float:
    """Calcula el solapamiento de n-gramas (aprox. tipo BLEU simplificado)."""
    hyp_toks = tokenize_simple(hyp)
    ref_toks = tokenize_simple(ref)
    if len(ref_toks) < n:
        return 0.0

    hyp_ngrams = Counter(tuple(hyp_toks[i:i+n]) for i in range(len(hyp_toks) - n + 1))
    ref_ngrams = Counter(tuple(ref_toks[i:i+n]) for i in range(len(ref_toks) - n + 1))

    overlap = sum((hyp_ngrams & ref_ngrams).values())
    total_ref = max(sum(ref_ngrams.values()), 1)
    return overlap / total_ref

# Pequeña prueba
print(ngram_overlap("this is a test", "this is a simple test", n=1))
print(ngram_overlap("this is a test", "this is a simple test", n=2))

0.8
0.5


In [ ]:
def eval_translations(hyps, refs):
    results = []
    for hyp, ref in zip(hyps, refs):
        uni = ngram_overlap(hyp, ref, n=1)
        bi = ngram_overlap(hyp, ref, n=2)
        results.append((uni, bi))
    return np.array(results)

scores_greedy = eval_translations(translations_greedy, refs)
print("Greedy — mean 1-gram:", scores_greedy[:, 0].mean())
print("Greedy — mean 2-gram:", scores_greedy[:, 1].mean())

scores_beam4 = eval_translations(translations_beam4, refs)
print("Beam=4 — mean 1-gram:", scores_beam4[:, 0].mean())
print("Beam=4 — mean 2-gram:", scores_beam4[:, 1].mean())

print("\nDetalle por frase (unigram, bigram):")
for i, (g, b4) in enumerate(zip(scores_greedy, scores_beam4)):
    print(f"Ejemplo {i+1}: Greedy={g}, Beam4={b4}")

Greedy — mean 1-gram: 0.7899999999999999
Greedy — mean 2-gram: 0.6642857142857143
Beam=4 — mean 1-gram: 0.7899999999999999
Beam=4 — mean 2-gram: 0.6642857142857143

Detalle por frase (unigram, bigram):
Ejemplo 1: Greedy=[1. 1.], Beam4=[1. 1.]
Ejemplo 2: Greedy=[0.8  0.75], Beam4=[0.8  0.75]
Ejemplo 3: Greedy=[0.75       0.57142857], Beam4=[0.75       0.57142857]
Ejemplo 4: Greedy=[0.5        0.11111111], Beam4=[0.5        0.11111111]
Ejemplo 5: Greedy=[0.9        0.88888889], Beam4=[0.9        0.88888889]


## 11) Análisis de errores típicos y conclusiones

En tus ejecuciones concretas pueden variar ligeramente los resultados (por sampling, versión de librerías, etc.), pero **típicamente** se observan patrones como:

- El beam search (`num_beams=4`) suele aumentar el solapamiento de bigramas respecto a greedy, aunque la diferencia no siempre es grande.
- En algunos ejemplos, greedy produce traducciones más cortas y a veces omite matices (under-translation).
- El modelo tiende a ser bastante literal; en frases más técnicas o de dominio muy específico puede cometer errores léxicos.

### Tipos de errores típicos observados

1. **Under-translation**  
   Ejemplo típico: se omite parte de una enumeración o un adjetivo importante, manteniendo solo la idea principal.

2. **Over-translation**  
   El modelo añade expresiones como *"in general"* o *"in particular"* que no estaban en el original, intentando hacer la frase más “natural”.

3. **Errores léxicos / de estilo**  
   - Traducciones demasiado literales de términos técnicos.
   - Cambios de registro (formal ↔ informal) que no estaban en la frase original.

Estos errores se relacionan con:

- La naturaleza probabilista del modelo y los datos de entrenamiento.
- Las decisiones de decodificación: beams más grandes tienden a favorecer traducciones más probables globalmente, pero también pueden penalizar traducciones más creativas o alternativas correctas.
- El hecho de que la mini-métrica de n-gramas (y BLEU en general) valora mucho la coincidencia superficial de palabras, pero no captura bien sinónimos ni reformulaciones correctas.

---

### Conclusión general

En este notebook has:

- Utilizado un modelo **encoder–decoder Transformer (MarianMT)** para traducir frases ES↔EN.
- Experimentado con **greedy**, **beam search** y **sampling**, observando su impacto en calidad y diversidad.
- Implementado una mini-métrica de evaluación basada en **n-gramas**, que ilustra la idea detrás de métricas como BLEU.
- Analizado errores típicos de NMT y conectado estos fenómenos con la teoría del Tema 4 (NMT).

Como extensión, podrías:

- Probar el modelo inverso (`en→es`) y comparar dificultades en cada dirección.
- Añadir más ejemplos de tu dominio (p.ej., textos de IA, optimización, etc.).
- Comparar MarianMT con otros modelos seq2seq (T5, mBART) en un conjunto de frases más amplio.
